[< Back to Demo 01](test_graphrag.ipynb) | [Demo README](README.md)

# Optional Demo 01b: Neo4j Retrieval Patterns

Once the data is in a graph, which retrieval pattern should I use? This optional notebook compares four high-level choices over the same hotel graph: vector, hybrid, Vector-Cypher, and Text2Cypher retrieval.

This notebook retrieves and displays evidence. It deliberately does **not** ask another LLM to turn that evidence into a final answer.

## Before you run this notebook

Run Demo 01 preparation from this directory. It builds the deterministic 30-document sample when needed, reuses the Nova embeddings written to `:Chunk.embedding`, and provisions both retrieval indexes:

```bash
uv run prepare_graph.py --mode lite
```

The preparation workflow is idempotent. When the graph and indexes are already ready, it reports readiness without rebuilding.

In [ ]:
import os

from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display
from neo4j import GraphDatabase
from neo4j_graphrag.retrievers import (
    HybridRetriever,
    Text2CypherRetriever,
    VectorCypherRetriever,
    VectorRetriever,
)
from neo4j_graphrag.types import RetrieverResultItem

load_dotenv()

from bedrock_providers import BedrockEmbeddings, BedrockLLM
from graph_config import GRAPH_SCHEMA, NEO4J_URI, neo4j_auth
from retrieval_contract import CHUNK_FULLTEXT_INDEX, CHUNK_VECTOR_INDEX
from retrieval_setup import fixture_problems, verify_retrieval_indexes

driver = GraphDatabase.driver(NEO4J_URI, auth=neo4j_auth())
driver.verify_connectivity()
print('Connected to Neo4j.')

## Verify the prepared graph

Retrieval notebooks should not create schema artifacts. This check requires both indexes to be online with the expected label, property, dimensions, and similarity function, then checks every demo-critical graph fixture.

In [ ]:
try:
    verify_retrieval_indexes(driver)
    problems = fixture_problems(driver)
    if problems:
        raise RuntimeError('; '.join(problems))
except Exception as exc:
    raise RuntimeError(
        f'Demo 01b is not ready: {exc}\n'
        'From 01-graphrag-demo, run: uv run prepare_graph.py --mode lite'
    ) from exc

print(f'✓ {CHUNK_VECTOR_INDEX} is online')
print(f'✓ {CHUNK_FULLTEXT_INDEX} is online')
print('✓ Demo-critical fixtures are present')

## The pinned hotel graph schema

The extraction pipeline uses one explicit schema. Render it before using a traversal-based retriever so the relationships in later context are predictable.

In [ ]:
pattern_rows = ''.join(
    f'<tr><td><strong>{source}</strong></td><td>—{relationship}→</td>'
    f'<td><strong>{target}</strong></td></tr>'
    for source, relationship, target in GRAPH_SCHEMA['patterns']
)
display(HTML(
    '<table><thead><tr><th>From</th><th>Relationship</th><th>To</th></tr>'
    f'</thead><tbody>{pattern_rows}</tbody></table>'
    '<p>Each extracted entity also points to its source '    '<code>(entity)-[:FROM_CHUNK]-&gt;(:Chunk)</code>.</p>'
))

## Shared query embedding contract

The graph build already wrote 1024-dimensional Amazon Nova 2 embeddings with the `GENERIC_INDEX` purpose. Query retrieval uses that same provider contract. It does not recompute or overwrite stored chunk embeddings.

In [ ]:
embedder = BedrockEmbeddings(region_name=os.environ.get('AWS_REGION', 'us-east-1'))

def show_results(question, result, why):
    print(f'Question: {question}\n')
    for number, item in enumerate(result.items, 1):
        score = (item.metadata or {}).get('score')
        score_text = 'n/a' if score is None else f'{score:.4f}'
        content = str(item.content)
        preview = content[:700] + ('…' if len(content) > 700 else '')
        print(f'[{number}] score={score_text}\n{preview}\n')
    print(f'Why this fits: {why}')

## Pattern 1: Vector retrieval for semantic lookup

Use vector retrieval when the learner's wording may differ from the source and the answer lives in a relevant chunk.

In [ ]:
vector_retriever = VectorRetriever(
    driver=driver,
    index_name=CHUNK_VECTOR_INDEX,
    embedder=embedder,
    return_properties=['text'],
)
vector_question = (
    'What is the standard check-in time at the AnyCompany Cairo Nile View hotel?'
)
vector_result = vector_retriever.search(
    query_text=vector_question,
    top_k=3,
)
show_results(
    vector_question,
    vector_result,
    'Semantic similarity finds the policy wording even when the question is paraphrased.',
)

## Pattern 2: Hybrid retrieval for exact names and identifiers

Embeddings can blur exact identifiers. In the deterministic lite sample, the chunk for Windward Mile Tower contains postal code `60611`; historical validation ranked it 12th with pure vector search. Compare a typical top five with hybrid retrieval, which uses the full question for its vector signal and the extracted postal code for its full-text signal before fusing the rankings.

In [ ]:
identifier_question = 'What is the cancellation policy for the hotel at 60611?'
vector_identifier_result = vector_retriever.search(
    query_text=identifier_question,
    top_k=5,
)

hybrid_retriever = HybridRetriever(
    driver=driver,
    vector_index_name=CHUNK_VECTOR_INDEX,
    fulltext_index_name=CHUNK_FULLTEXT_INDEX,
    embedder=embedder,
    return_properties=['text'],
)
hybrid_result = hybrid_retriever.search(
    query_text='60611',
    query_vector=vector_identifier_result.metadata['query_vector'],
    top_k=5,
    ranker='linear',
    alpha=0.2,
)

show_results(
    identifier_question,
    vector_identifier_result,
    'This is the semantic-only comparison. An exact postal code is not a strong semantic signal.',
)
print('\n' + '=' * 80 + '\n')
show_results(
    identifier_question,
    hybrid_result,
    'Full-text matching preserves 60611 while the separately supplied question vector handles the policy wording.',
)
assert any('60611' in str(item.content) for item in hybrid_result.items)

## Pattern 3: Vector-Cypher for graph-enriched lookup

Use Vector-Cypher when semantic similarity should identify a chunk first, then graph traversal should add structured hotel and amenity context.

In [ ]:
retrieval_query = '''
MATCH (hotel:Hotel)-[:FROM_CHUNK]->(node)
OPTIONAL MATCH (hotel)-[relationship]->(detail)
WHERE type(relationship) IN [
    'HAS_ROOM', 'OFFERS_AMENITY', 'HAS_POLICY', 'PROVIDES_SERVICE'
]
WITH node, score, hotel,
     collect(DISTINCT {
         relationship: type(relationship),
         name: coalesce(detail.name, detail.type),
         description: detail.description
     })[..12] AS related
RETURN node.text AS chunk, score,
       hotel { .name, .address, .guest_rating } AS hotel,
       related
'''

def graph_result_formatter(record):
    content = {
        'chunk': record.get('chunk'),
        'hotel': record.get('hotel'),
        'related': record.get('related'),
    }
    return RetrieverResultItem(
        content=str(content),
        metadata={'score': record.get('score')},
    )

vector_cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name=CHUNK_VECTOR_INDEX,
    retrieval_query=retrieval_query,
    embedder=embedder,
    result_formatter=graph_result_formatter,
)
graph_question = (
    'Tell me about the hotel at 789 Avenue des Champs-Élysées and its amenities.'
)
graph_result = vector_cypher_retriever.search(
    query_text=graph_question,
    top_k=2,
)
show_results(
    graph_question,
    graph_result,
    'Vector search locates source chunks; Cypher adds connected hotel entities and typed relationships.',
)

## Pattern 4: Text2Cypher for counts and flexible structured questions

Use Text2Cypher when the question needs aggregation, filtering, or multi-hop structure rather than top-k chunks. Display the generated query and database records directly.

In [ ]:
hotel_schema = '''
Node properties:
Hotel {name: STRING, address: STRING, guest_rating: FLOAT, total_rooms: INTEGER}
Room {type: STRING, bed_configuration: STRING, max_occupancy: INTEGER}
Amenity {name: STRING, description: STRING, fee: STRING}
Policy {name: STRING, description: STRING}
Service {name: STRING, description: STRING, cost: STRING, hours: STRING}
Relationships:
(:Hotel)-[:HAS_ROOM]->(:Room)
(:Hotel)-[:OFFERS_AMENITY]->(:Amenity)
(:Hotel)-[:HAS_POLICY]->(:Policy)
(:Hotel)-[:PROVIDES_SERVICE]->(:Service)
'''
examples = [
    "USER INPUT: What is the average rating of Paris hotels? CYPHER: MATCH (h:Hotel) WHERE toLower(h.address) CONTAINS 'paris' AND h.guest_rating IS NOT NULL RETURN avg(h.guest_rating) AS average_rating",
    "USER INPUT: How many hotels have a spa? CYPHER: MATCH (h:Hotel)-[:OFFERS_AMENITY]->(a:Amenity) WHERE toLower(a.name) CONTAINS 'spa' RETURN count(DISTINCT h) AS hotel_count",
]
text2cypher_prompt = '''
Generate one read-only Cypher 25 query for the user question.
Use only the supplied schema. Never write or delete data.
Return only the Cypher query with no markdown fence or explanation.
Schema:
{schema}
Examples:
{examples}
User question: {query_text}
'''
text2cypher_retriever = Text2CypherRetriever(
    driver=driver,
    llm=BedrockLLM(region_name=os.environ.get('AWS_REGION', 'us-east-1')),
    neo4j_schema=hotel_schema,
    examples=examples,
    custom_prompt=text2cypher_prompt,
)
structured_question = 'How many hotels in the database have a swimming pool?'
structured_result = text2cypher_retriever.search(
    query_text=structured_question
)

print(f'Question: {structured_question}')
print(f"Generated Cypher:\n{structured_result.metadata['cypher']}\n")
print('Returned records:')
for item in structured_result.items:
    print(f'  {item.content}')
print('\nWhy this fits: the database computes the count over all matching relationships.')

### The same pattern behind a trust boundary

This section teaches the library-level Text2Cypher pattern: `Text2CypherRetriever` runs in-process in this notebook, holds raw database credentials, and receives the hand-pinned `hotel_schema`. The optional advanced [Demo 09](../09-neo4j-mcp-demo/) runs the same pattern through a pre-deployed read-only Neo4j MCP service, where schema pinning and read-only enforcement move server-side. That is a trust-boundary change rather than a capability change: the agent calls a governed tool instead of holding credentials. Demo 06A takes a different route and productionizes the fixed Hybrid-Cypher pattern.

## Going further: HybridCypherRetriever

`HybridCypherRetriever` combines the exact-match strength of hybrid search with the graph expansion used by Vector-Cypher. It is useful when a query contains an identifier and also needs connected context. Keep it as an advanced variation, not a fifth main pattern.

## Which pattern should I use?

| Query shape | Start with | Why |
|---|---|---|
| Semantic lookup or paraphrase | VectorRetriever | Meaning matters more than exact wording |
| Exact hotel name, policy term, or identifier | HybridRetriever | Combines semantic and full-text signals |
| Semantic lookup plus connected hotel context | VectorCypherRetriever | Retrieves a chunk, then traverses relationships |
| Aggregation or count | Text2CypherRetriever | Lets Neo4j calculate over the full matching set |
| Flexible multi-hop structured question | Text2CypherRetriever | Generates a read-only traversal against the pinned schema |
| Question outside the graph | No retriever can invent coverage | Return an explicit empty result or abstain |

## Chunking note

This workshop uses large chunks during graph extraction so each hotel stays intact while entities and relationships are created. Production systems may keep that extraction strategy and create separate, smaller retrieval-oriented chunks. The right retrieval chunk size depends on the source material and query shapes; it is not tuned in this high-level workshop.

In [ ]:
driver.close()
print('Connection closed.')